In [3]:
# 訓練データとテストデータの画像を読み込む
# （サイズは縦横224pxにリサイズする）
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

# 分類名（no／yes）をリストとして格納する
class_names = train_dataset.class_names
class_names


# 画像の水増しをする関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

# 画像の水増し処理の実行
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

# 水増ししたデータを訓練データに追加する
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

# データをシャッフルする
train_dataset = train_dataset.shuffle(32)

# MobileNetV2モデルを作成する
input_layer = tf.keras.Input(shape=(224, 224, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

# Dense層を追加する
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')

# base_modelに先ほどのDense層を追加したモデルを作成する
model = tf.keras.Sequential([
    base_model,
    output_layer
])

# modelをcompileする
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

# modelに学習させる
model.fit(train_dataset, epochs=20)

# テストデータで分類を実行する
pred_data = model.predict(test_dataset)
pred_data

# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)



Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 43s 504ms/step - accuracy: 0.8217 - loss: 0.4032
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 31s 496ms/step - accuracy: 0.9328 - loss: 0.2089
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 47s 768ms/step - accuracy: 0.9517 - loss: 0.1581
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 78s 666ms/step - accuracy: 0.9650 - loss: 0.1310
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 35s 559ms/step - accuracy: 0.9722 - loss: 0.1119
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 37s 591ms/step - accuracy: 0.9783 - loss: 0.0937
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 35s 554ms/step - accuracy: 0.9789 - loss: 0.0833
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 36s 585ms/step - accuracy: 0.9811 - loss: 0.0748
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 31s 497ms/step - accuracy: 0.9850 - loss: 0.0682
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 35s 569ms/step - accuracy: 0.9878 - loss: 0.0611
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 35s 569ms/

[0.0794653668999672, 0.9599999785423279]